# 08-statement-logistic
## Ответ: для обычной: 0.927, для L2: 0.936

### 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения-1 или 1.
### 2. Убедитесь, что  выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

In [2]:
import pandas as pd
import math
from sklearn.metrics import roc_auc_score

url = 'https://raw.githubusercontent.com/Sazankova/logistic/main/data-logistic.csv'
data = pd.read_csv(url, header=None)

X = data[[1, 2]]
y = data[0]

def e_distance(q1, q2, p1, p2):
    return ((q1 - p1)**2 + (q2 - p2)**2)**0.5

def function(i, y, X, w1, w2, j):
    return y.iloc[i] * X.iloc[i, j] * (1 - 1 / (1 + math.exp(-y[i] * (w1 * X.iloc[i, 0] + w2 * X.iloc[i, 1]))))


### 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0). 
### 4.Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.
### 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 +exp(−w1x1 −w2x2)).

### для обычной функции

In [3]:
# 3. Обычная логистическая регрессия (без регуляризации)
def gradient(w1, w2):
    w1_new = w1 + k * (1 / l) * sum(function(i, y, X, w1, w2, 0) for i in range(l))
    w2_new = w2 + k * (1 / l) * sum(function(i, y, X, w1, w2, 1) for i in range(l))
    return (w1_new, w2_new)

l = len(X)
k = 0.1
e = 1e-5
w1, w2 = 0, 0

iteration = 1
while iteration < 10000:
    w_vector = gradient(w1, w2)
    if e_distance(w1, w2, w_vector[0], w_vector[1]) < e:
        w1 = w_vector[0]
        w2 = w_vector[1]
        print("Сходимость достигнута, итерация:", iteration)
        break
    else:
        w1 = w_vector[0]
        w2 = w_vector[1]
        iteration += 1

p = [1 / (1 + math.exp(-(w1 * X.iloc[i, 0] + w2 * X.iloc[i, 1]))) for i in range(l)]
y_binary = [1 if yi == 1 else 0 for yi in y]

roc_auc = roc_auc_score(y_binary, p)
round(roc_auc, 3)

Сходимость достигнута, итерация: 244


0.927

### для L2-регуляризованной

In [4]:
# 4. L2-регуляризованная логистическая регрессия (C=10)
w1_L2, w2_L2 = 0, 0
iteration_L2 = 1

def gradient_L2(w1_L2, w2_L2):
    w1_L2_new = w1_L2 + k * (1 / l) * sum(function(i, y, X, w1_L2, w2_L2, 0) for i in range(l)) - k * 10 * w1_L2
    w2_L2_new = w2_L2 + k * (1 / l) * sum(function(i, y, X, w1_L2, w2_L2, 1) for i in range(l)) - k * 10 * w2_L2
    return (w1_L2_new, w2_L2_new)

while iteration_L2 < 10000:
    w_vector_L2 = gradient_L2(w1_L2, w2_L2)
    if e_distance(w1_L2, w2_L2, w_vector_L2[0], w_vector_L2[1]) < e:
        w1_L2 = w_vector_L2[0]
        w2_L2 = w_vector_L2[1]
        print("Сходимость достигнута, итерация:", iteration_L2)
        break
    else:
        w1_L2 = w_vector_L2[0]
        w2_L2 = w_vector_L2[1]
        iteration_L2 += 1

p_L2 = [1 / (1 + math.exp(-(w1_L2 * X.iloc[i, 0] + w2_L2 * X.iloc[i, 1]))) for i in range(l)]

roc_auc_L2 = roc_auc_score(y_binary, p_L2)
round(roc_auc_L2, 3)

Сходимость достигнута, итерация: 8


0.936

### 6. Влияние длины шага: При увеличении длины шага количество итераций уменьшается. При k=0.1 потребовалось 244 итерации, при k=0.5 — 60 итераций, при k=0.7 — 40 итераций, при k=1.0 — 32 итерации, при k=1.1 — 29 итераций. При k=2.0 алгоритм не сошелся. Таким образом, при уменьшении длины шага количество итераций увеличивается. Слишком большой шаг может привести к расходимости алгоритма.

### 7. Влияние начального приближения: При начальных весах (10, 20) потребовалось 723 итерации, при (-10, -20) — 606 итераций, при (0, 0) — 244 итерации. Начальное приближение влияет только на скорость сходимости (количество итераций), но не на итоговое качество модели. Значение AUC-ROC при разных начальных приближениях остается одинаковым.